# Credit Card Default Risk Model & Validation

**Business problem:** A card issuer loses money when customers default. This project predicts each customer's Probability of Default (PD), validates the model like a bank's model-risk team would, and shows how a cut-off affects approvals, bad rate and profit.

**Data:** UCI "Default of Credit Card Clients" (30,000 accounts, Taiwan). Target = defaulted next month.

## 1. First model
Created three features: credit utilization, payment ratio and maximum payment delay. Fitted a logistic regression (explainable, standard in banks). The cut-off table below uses placeholder profit values (+100 per good account, -500 per bad account) and is from this first model.

In [8]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

df = pd.read_csv("UCI_Credit_Card.csv")
df["util"] = df["BILL_AMT1"] / df["LIMIT_BAL"]
df["pay_ratio"] = df["PAY_AMT1"] / (df["BILL_AMT1"].abs() + 1)
df["max_delay"] = df[["PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]].max(axis=1)

X = df[["LIMIT_BAL","AGE","util","pay_ratio","max_delay","PAY_0"]]
y = df["default.payment.next.month"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
p = model.predict_proba(Xte)[:, 1]

auc = roc_auc_score(yte, p)
fpr, tpr, _ = roc_curve(yte, p)
print("AUC:", auc, "Gini:", 2*auc-1, "KS:", max(tpr-fpr))

def psi(expected, actual, bins=10):
    cuts = np.percentile(expected, np.linspace(0, 100, bins+1))
    cuts[0], cuts[-1] = -np.inf, np.inf
    e = np.histogram(expected, cuts)[0] / len(expected) + 1e-6
    a = np.histogram(actual, cuts)[0] / len(actual) + 1e-6
    return np.sum((a-e) * np.log(a/e))

# Strategy: cut-off table
out = pd.DataFrame({"p": p, "bad": yte.values})
for c in [0.1, 0.2, 0.3, 0.4, 0.5]:
    approved = out[out.p < c]
    profit = (len(approved[approved.bad==0]) * 100) - (len(approved[approved.bad==1]) * 500)
    print(c, len(approved)/len(out), approved.bad.mean(), profit)

AUC: 0.7235719533735738 Gini: 0.44714390674714766 KS: 0.38471258772623473
0.1 0.1922222222222222 0.12658959537572254 41600
0.2 0.6436666666666667 0.1220438460210599 155100
0.3 0.7472222222222222 0.13516728624535315 127100
0.4 0.8574444444444445 0.15563042633147597 51100
0.5 0.931 0.18546365914786966 -94500


## 2. Validating the first model
Calibration compares predicted vs actual default rate in each risk decile.

**Finding:** the safest decile was under-predicted (4.4% vs 13.7% actual) and the middle deciles were over-predicted. The model ranks well (Gini 45%) but is not well calibrated.

In [9]:
out["decile"] = pd.qcut(out.p, 10, labels=False, duplicates="drop")
calib = out.groupby("decile").agg(avg_pred=("p","mean"), actual=("bad","mean"), count=("bad","size"))
print(calib)

        avg_pred    actual  count
decile                           
0       0.044409  0.136667    900
1       0.087205  0.116667    900
2       0.117516  0.127778    900
3       0.142969  0.090000    900
4       0.161922  0.123333    900
5       0.178141  0.123333    900
6       0.208129  0.175556    900
7       0.298976  0.251111    900
8       0.394446  0.391111    900
9       0.573588  0.676667    900


## 3. Fixing the problem: PAY_0 as a category
PAY_0 is repayment status (-2, -1, 0, 1, 2... months late). It is a category, not a straight-line number. I converted it to dummy variables, grouping 4+ months late together.

**Result:** Gini 45% to 49%, KS 38% to 39%, and predicted vs actual default rates now match across all deciles.

In [10]:
X2 = df[["LIMIT_BAL","AGE","util","pay_ratio","max_delay"]].copy()
dummies = pd.get_dummies(df["PAY_0"].clip(upper=4), prefix="PAY0", dtype=int)
X2 = pd.concat([X2, dummies], axis=1)

Xtr2, Xte2, ytr2, yte2 = train_test_split(X2, y, test_size=0.3, stratify=y, random_state=42)
model2 = LogisticRegression(max_iter=5000).fit(Xtr2, ytr2)
p2 = model2.predict_proba(Xte2)[:, 1]

auc2 = roc_auc_score(yte2, p2)
fpr2, tpr2, _ = roc_curve(yte2, p2)
print("AUC:", auc2, "Gini:", 2*auc2-1, "KS:", max(tpr2-fpr2))

out2 = pd.DataFrame({"p": p2, "bad": yte2.values})
out2["decile"] = pd.qcut(out2.p, 10, labels=False, duplicates="drop")
print(out2.groupby("decile").agg(avg_pred=("p","mean"), actual=("bad","mean"), count=("bad","size")))

AUC: 0.7449473551225916 Gini: 0.48989471024518316 KS: 0.38869125646662656
        avg_pred    actual  count
decile                           
0       0.073362  0.073333    900
1       0.094917  0.110000    900
2       0.108464  0.102222    900
3       0.119537  0.134444    900
4       0.129838  0.124444    900
5       0.143940  0.168889    900
6       0.178871  0.176667    900
7       0.247872  0.240000    900
8       0.397750  0.393333    900
9       0.698665  0.688889    900


## 4. Data quality check
No missing values; default rate 22.1%. EDUCATION and MARRIAGE contain undocumented codes (about 1% of accounts). These variables are not used in the model, so they were left unchanged.

In [11]:
print(df.shape)
print(df.isnull().sum().sum(), "missing values")
print("Default rate:", df["default.payment.next.month"].mean())
print(df["EDUCATION"].value_counts())
print(df["MARRIAGE"].value_counts())


(30000, 28)
0 missing values
Default rate: 0.2212
EDUCATION
2    14030
1    10585
3     4917
5      280
4      123
6       51
0       14
Name: count, dtype: int64
MARRIAGE
2    15964
1    13659
3      323
0       54
Name: count, dtype: int64


## 5. Conclusion and limitations
The final model has a Gini of 49% and KS of 39%, with good calibration across deciles after treating repayment status as a category.

**Limitations and future work:** one country and one time period; profit values are assumptions; no macroeconomic variables; no reject inference. Not done: PSI stability testing, a challenger model (decision tree) and ongoing monitoring (the dataset has no dates).